In [ ]:
import time
start_time = time.time()

In [ ]:
!pip install transformers datasets torch pandas scikit-learn openpyxl --quiet

In [ ]:
import pandas as pd
import torch
import numpy as np
import random
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
import logging
from transformers import logging as hf_logging
hf_logging.set_verbosity_error()

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_seed(42)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
file_path = "/content/drive/MyDrive/FYP data set.xlsx"
df = pd.read_excel(file_path)
print("File loaded successfully!")

In [ ]:
def clean_label(label):
    return str(label).strip().lower().replace("-", "").replace("\t", "").replace("  ", " ")

df['CleanGroup'] = df['Group'].apply(clean_label)

# Drop missing/invalid labels
df = df[df['CleanGroup'].notna()]
df = df[~df['CleanGroup'].isin(["nan", "", "none", "null"])]

# Drop rows with empty text
df = df.dropna(subset=['Description'])
df = df[df['Description'].astype(str).str.strip() != ""]

# Encode target labels
label_encoder = LabelEncoder()
df['label'] = label_encoder.fit_transform(df['CleanGroup'])

print("Label Mapping:")
for idx, label in enumerate(label_encoder.classes_):
    print(f"{idx}: {label}")

In [ ]:
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df['Description'].astype(str).tolist(),
    df['label'].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

In [ ]:
model_name = "dmis-lab/biobert-base-cased-v1.1"
tokenizer = AutoTokenizer.from_pretrained(model_name)
MAX_TOKEN_LENGTH = 512
train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=MAX_TOKEN_LENGTH)
test_encodings = tokenizer(test_texts, truncation=True, padding=True, max_length=MAX_TOKEN_LENGTH)

train_dataset = Dataset.from_dict({
    'input_ids': train_encodings['input_ids'],
    'attention_mask': train_encodings['attention_mask'],
    'labels': train_labels
})

test_dataset = Dataset.from_dict({
    'input_ids': test_encodings['input_ids'],
    'attention_mask': test_encodings['attention_mask'],
    'labels': test_labels
})

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(label_encoder.classes_)
)

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    save_strategy="no",
    logging_dir="./logs",
    learning_rate=5e-5,
    seed=42,
    data_seed=42,
    report_to="none"
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

In [ ]:
trainer.train()

In [ ]:
predictions = trainer.predict(test_dataset)
pred_labels = predictions.predictions.argmax(-1)
print("\nClassification Report:\n")
print(classification_report(test_labels, pred_labels, target_names=label_encoder.classes_, zero_division=0))
print(f"\nModel Accuracy: {accuracy_score(test_labels, pred_labels)*100:.2f}%")

In [ ]:
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve
from torch.utils.data import DataLoader

# ==========================================
# 1. Temperature Scaling Class
# ==========================================
class TemperatureScaler(nn.Module):
    def __init__(self):
        super(TemperatureScaler, self).__init__()
        self.temperature = nn.Parameter(torch.ones(1) * 1.5)

    def forward(self, logits):
        return logits / self.temperature

    def set_temperature(self, eval_dataloader, model):
        model.eval()
        self.to(model.device)
        logits_list, labels_list = [], []

        with torch.no_grad():
            for batch in eval_dataloader:
                inputs = {k: v.to(model.device) for k, v in batch.items() if k != 'labels'}
                logits = model(**inputs).logits
                logits_list.append(logits)
                labels_list.append(batch['labels'])

        logits = torch.cat(logits_list).to(model.device)
        labels = torch.cat(labels_list).to(model.device)
        optimizer = optim.LBFGS([self.temperature], lr=0.01, max_iter=50)
        criterion = nn.CrossEntropyLoss()

        def eval():
            optimizer.zero_grad()
            loss = criterion(self.forward(logits), labels)
            loss.backward()
            return loss

        optimizer.step(eval)
        return self.temperature.item()

# ==========================================
# 2. Calibration & Plotting Execution
# ==========================================
# Setup Data
# Ensure test_dataset returns PyTorch tensors for DataLoader
test_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
eval_loader = DataLoader(test_dataset, batch_size=8)
scaler = TemperatureScaler()
optimal_t = scaler.set_temperature(eval_loader, model)
print(f"Optimal Temperature: {optimal_t:.4f}")

# Get Predictions
predictions = trainer.predict(test_dataset)
raw_logits = torch.tensor(predictions.predictions)

# Calculate Probabilities
raw_probs = torch.nn.functional.softmax(raw_logits, dim=-1).numpy()
calibrated_probs = torch.nn.functional.softmax(scaler(raw_logits.to(model.device)), dim=-1).cpu().detach().numpy()

# Plotting
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

def plot_on_ax(ax, probs, title, color):
    max_probs = np.max(probs, axis=1)
    preds = np.argmax(probs, axis=1)
    accs = (preds == test_labels).astype(int)
    prob_true, prob_pred = calibration_curve(accs, max_probs, n_bins=10)

    ax.plot([0, 1], [0, 1], "k--", label="Perfectly Calibrated")
    ax.plot(prob_pred, prob_true, "s-", color=color, label="BioBERT")
    ax.set_title(title)
    ax.set_xlabel("Confidence")
    ax.set_ylabel("Accuracy")
    ax.legend(loc="lower right")
    ax.grid(True, linestyle=':', alpha=0.6)

plot_on_ax(ax1, raw_probs, "Reliability Diagram - BioBERT (Raw)", "#1f77b4")
plot_on_ax(ax2, calibrated_probs, f"Reliability Diagram - BioBERT (Calibrated T={optimal_t:.2f})", "#2ca02c")

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
from sklearn.utils import resample

def plot_bootstrapped_confusion_matrix(y_true, y_probs, classes, n_iterations=1000):
    # Convert y_true to a numpy array to enable advanced indexing
    y_true = np.asarray(y_true)

    # Store matrices from each iteration
    matrices = []

    # 1. Bootstrapping loop
    for i in range(n_iterations):
        # Resample the test data with replacement
        indices = resample(np.arange(len(y_true)), replace=True)
        resample_true = y_true[indices]
        resample_probs = y_probs[indices]
        resample_preds = np.argmax(resample_probs, axis=1)

        # Calculate matrix for this sample
        cm = confusion_matrix(resample_true, resample_preds, labels=range(len(classes)))
        matrices.append(cm)

    # 2. Calculate Mean and Standard Deviation (CI)
    matrices = np.array(matrices)
    mean_cm = np.mean(matrices, axis=0)
    std_cm = np.std(matrices, axis=0)

    # 3. Create the Labels (Mean ± CI)
    # This creates the text inside each box: "1376.7 ± 51.0"
    annot_labels = np.empty_like(mean_cm, dtype=object)
    for i in range(mean_cm.shape[0]):
        for j in range(mean_cm.shape[1]):
            annot_labels[i, j] = f"{mean_cm[i, j]:.1f} ± {std_cm[i, j]:.1f}"

    # 4. Plotting
    plt.figure(figsize=(12, 10))
    sns.set_context("paper", font_scale=1.2)

    sns.heatmap(mean_cm, annot=annot_labels, fmt="", cmap='Blues',
                xticklabels=classes, yticklabels=classes,
                cbar_kws={'label': 'Mean Frequency'})

    plt.title('Confusion Matrix (mean ± CI) - BioBERT', pad=20, fontsize=15)
    plt.ylabel('True Category', fontsize=12)
    plt.xlabel('Predicted Category', fontsize=12)
    plt.xticks(rotation=45)

    plt.tight_layout()
    plt.show()

# Run using your calibrated results
plot_bootstrapped_confusion_matrix(test_labels, calibrated_probs, label_encoder.classes_)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

def plot_styled_confusion_matrix(y_true, y_pred, classes):
    # 1. Calculate Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)

    # 2. Setup Plot
    plt.figure(figsize=(10, 8))
    sns.set_context("paper", font_scale=1.2)

    # 3. Create Heatmap
    # 'fmt'='d' for integers, 'cmap'='Blues' matches professional standards
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=classes, yticklabels=classes,
                cbar_kws={'label': 'Number of Reports'})

    # 4. Labeling
    plt.title('Confusion Matrix: BioBERT MRI Classification', pad=20, fontsize=15)
    plt.ylabel('Actual Clinical Label', fontsize=12)
    plt.xlabel('Predicted Label', fontsize=12)
    plt.xticks(rotation=45)
    plt.yticks(rotation=0)

    plt.tight_layout()
    plt.show()

# Run using your existing test data
plot_styled_confusion_matrix(test_labels, pred_labels, label_encoder.classes_)

In [ ]:
# -----------------------------
# Record end time and print elapsed time
# -----------------------------
end_time = time.time()
elapsed_time = end_time - start_time
print(f"\n⏱️ Total execution time: {elapsed_time:.2f} seconds")

In [ ]:
save_path = "/content/drive/MyDrive/BioBERT_MRI_Model"
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f"✅ Model backed up to Google Drive at: {save_path}")

In [ ]:
import joblib
# Save the label encoder to your Google Drive folder
joblib.dump(label_encoder, "/content/drive/MyDrive/BioBERT_MRI_Model/label_encoder.joblib")
print("✅ Label Encoder saved successfully!")